In [1]:
# !uv add 'transformers[torch]' datasets evaluate

In [2]:
from huggingface_hub import notebook_login

notebook_login()

## Load Dataset

In [3]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

In [4]:
eli5 = eli5.train_test_split(test_size=0.2)
eli5 = eli5.flatten()

In [5]:
eli5["train"][0]

{'q_id': '5pima1',
 'title': 'They say that if you jumped off the empire state building, you would be dead before you hit the ground. Is this true and why??',
 'selftext': '',
 'category': 'Physics',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dcrjfn6', 'dcrgijz'],
 'answers.text': ["It's not true there is nothing that would cause you to doe before you hit the ground. The myth is based on the idea of dying of fright which isn't actually a thing. It's possible the jumper might have a heart attack on the way down but that same person was likely prone to a heart attack already and would have had one very shortly anyway and there likely wouldn't be enough time for them to actually die of the heart attack before they hit even if it started when they jumped. If this were true skydiving would be far more fatal than it is and would likely be illegal.",
  "I believe this is false for anyone in decent physical health. Skydivers in free fall routinely reach terminal velocity, which ofte

## Load Model

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

In [7]:
tokenizer

GPT2Tokenizer(name_or_path='distilbert/distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [8]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [9]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    remove_columns=eli5["train"].column_names,
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1433 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### Spliting into 128 lengthed sentences
This is done to fit things into gpu memory.

In [10]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [11]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

In [13]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [15]:
training_args = TrainingArguments(
    output_dir="eli5_clm-model-invision",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.919402,3.798566
2,3.819770,3.787037
3,3.789831,3.784517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3987, training_loss=3.847231254261898, metrics={'train_runtime': 477.7863, 'train_samples_per_second': 66.714, 'train_steps_per_second': 8.345, 'total_flos': 1041104240640000.0, 'train_loss': 3.847231254261898, 'epoch': 3.0})

In [16]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
3.789831,3.784517,3


Perplexity: 44.01


In [17]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/abhishekkapoor/eli5_clm-model-invision/commit/e55d7dc11c82c82f8620c8dcabeed638727a1ad4', commit_message='End of training', commit_description='', oid='e55d7dc11c82c82f8620c8dcabeed638727a1ad4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/abhishekkapoor/eli5_clm-model-invision', endpoint='https://huggingface.co', repo_type='model', repo_id='abhishekkapoor/eli5_clm-model-invision'), pr_revision=None, pr_num=None)

## Inference

In [18]:
prompt = "Somatic hypermutation allows the immune system to"

In [19]:
from transformers import pipeline

generator = pipeline("text-generation", model="abhishekkapoor/eli5_clm-model-invision")
generator(prompt)

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Somatic hypermutation allows the immune system to detect the presence of pathogens at the base of the cell and then it would know which cells are infecting the cells to fight off the infection. This way, the immune system would be able to detect which cells are most likely to be infected. This is how it works. If your immune system detects a threat that you are a part of, it then detects it as a threat. This allows the immune system to detect viruses that could be infected. If your immune system detects a virus or infection that is causing you to develop symptoms, the virus that you have detected is just the virus you are infected. If you are infected by a virus that is causing you to develop symptoms, the virus that you have detected is just the virus you are infected. The immune system can detect any threat to your immune system, but this is not what you are expected to notice. For example, there is a person who is infected by an infected person who is a member o

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("abhishekkapoor/eli5_clm-model-invision")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

In [22]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("abhishekkapoor/eli5_clm-model-invision")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [23]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

["Somatic hypermutation allows the immune system to adapt to a more drastic response. When the immune system is overwhelmed, the immune system does NOT adapt to new stimuli (such as allergies, etc...) The reason is that the immune system reacts to these changes more quickly than before (which is why it will do so by limiting the number of times the immune system gets affected by the food/whatever reaction). When a food is served, however, there is an immune response to an event in another food. It's just that that, so you have"]